# Blackpolis: Systematic Racism in Cali — Replication Notebook

Quantitative replication of the policy brief *Blackpolis: Systematic Racism in Cali* (Daniel Navarro). The brief analyses the socioeconomic gap between Afro-Colombian (NARP) communities and the rest of the population in Cali, using neighborhood-level data from the 2018 national census and CaliEnDatos.

The pipeline:
1. Load the cleaned, merged neighborhood dataset.
2. Inspect correlations between ethnicity, education, literacy, unemployment and socioeconomic stratum.
3. Fit bivariate linear regressions of each socioeconomic outcome on Afro ethnicity share.
4. Apply k-means clustering (k = 3) to group neighborhoods by socioeconomic profile.
5. Inspect cluster composition and relate it back to ethnicity.

## 1. Imports

In [ ]:
import math
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import pearsonr, spearmanr
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression

warnings.filterwarnings('ignore')

DATA_DIR = Path('data')
FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)

## 2. Load data

`datamerge_clustered.csv` is the cleaned, neighborhood-level dataset (one row per *barrio* in Cali). It already merges the 2018 census variables with NARP self-identification share. The original raw files (separate census extracts and the NARP file) were lost, so the analysis starts from this cleaned snapshot.

In [ ]:
data = pd.read_csv(DATA_DIR / 'datamerge_clustered.csv')
data.head()

Rename the Spanish columns to English for readability.

In [ ]:
rename_map = {
    'idbarrio#': 'neighborhood_id',
    'namebarrio': 'neighborhood_name',
    'id_barrio': 'neighborhood_code',
    'No lee %': 'does_not_read',
    'Asiste a Educación 5-21 años': 'attends_education_5_21',
    'Asiste a Educación superior': 'attends_higher_education',
    'Desempleo': 'unemployment',
    'Estrato': 'stratum',
    'barrio': 'barrio',
    'Pertenenci': 'ethnicity_afro',
    'Cluster': 'cluster_saved',
}
data = data.rename(columns=rename_map)

if 'cluster' in data.columns and 'cluster_saved' in data.columns:
    data = data.drop(columns=['cluster'])

data.info()

Variable definitions:
- `neighborhood_id` / `neighborhood_code` / `neighborhood_name` / `barrio`: identifiers and labels.
- `does_not_read`: share of residents who cannot read (0–1).
- `attends_education_5_21`: share of residents aged 5–21 attending any form of education.
- `attends_higher_education`: share of residents attending or having attended higher education.
- `unemployment`: share of residents unemployed (looking for work).
- `stratum`: Colombian socioeconomic stratum (1–6), assigned by the municipality.
- `ethnicity_afro`: share of residents self-identified as Black, Afro-Colombian, Afro-descendant, *Raizal* or *Palenquero* (NARP). Independent variable.
- `cluster_saved`: cluster assignment from a previous k-means run, retained for reference.

In [ ]:
data.describe(include='all')

## 3. Correlation analysis

Three heatmaps: Pearson (linear) and Spearman (rank) significance, plus a full correlation matrix with magnitudes. Asterisks encode p-value bands: `***` < 0.001, `**` < 0.01, `*` < 0.05.

In [ ]:
id_cols = ['neighborhood_id', 'neighborhood_name', 'neighborhood_code', 'barrio', 'cluster_saved']
numeric = data.drop(columns=[c for c in id_cols if c in data.columns])
numeric = numeric.apply(pd.to_numeric, errors='coerce').dropna()
col_names = numeric.columns.tolist()

def significance_heatmap(numeric_df, corr_fn, title, save_as=None):
    n = numeric_df.shape[1]
    corr_matrix = np.zeros((n, n))
    p_vals = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            corr_ij, p_val_ij = corr_fn(numeric_df.iloc[:, i], numeric_df.iloc[:, j])
            corr_matrix[i, j] = corr_ij
            p_vals[i, j] = p_val_ij

    mask = np.zeros_like(corr_matrix, dtype=bool)
    mask[np.triu_indices_from(mask)] = True

    sig = np.empty_like(p_vals, dtype='<U5')
    sig[p_vals < 0.001] = '***'
    sig[(p_vals >= 0.001) & (p_vals < 0.01)] = '**'
    sig[(p_vals >= 0.01) & (p_vals < 0.05)] = '*'
    sig[p_vals >= 0.05] = ''

    sns.set(font_scale=1)
    cmap = sns.diverging_palette(10, 250, as_cmap=True)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        corr_matrix, mask=mask, cmap=cmap, annot=sig, fmt='',
        vmin=-1, vmax=1, square=True, linewidths=.5,
        cbar_kws={'shrink': .5},
        xticklabels=numeric_df.columns, yticklabels=numeric_df.columns, ax=ax,
    )
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha='right')
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, ha='right')
    ax.set_title(title)
    plt.tight_layout()
    if save_as is not None:
        plt.savefig(save_as, dpi=150, bbox_inches='tight')
    plt.show()
    return corr_matrix, p_vals

_ = significance_heatmap(numeric, pearsonr, 'Pearson correlation — significance')

In [ ]:
_ = significance_heatmap(numeric, spearmanr, 'Spearman correlation — significance')

All pairwise correlations come out highly significant under both Pearson and Spearman. This is partly an artifact of all variables coming from the same census, so the next heatmap focuses on the *magnitude* and sign of the correlations rather than p-values.

In [ ]:
for outcome in ['unemployment', 'does_not_read', 'attends_education_5_21']:
    corr, p = pearsonr(numeric['ethnicity_afro'], numeric[outcome])
    print(f'ethnicity_afro vs {outcome}: r = {corr:.4f}, p = {p:.3e}')

In [ ]:
corr = numeric.corr()
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
cmap = sns.diverging_palette(10, 250, as_cmap=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr, mask=mask, cmap=cmap, annot=True, fmt='.2f',
    vmin=-1, vmax=1, square=True, linewidths=.5,
    cbar_kws={'shrink': .5}, ax=ax,
)
ax.set_title('Correlation matrix (English labels)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

Key correlations with `ethnicity_afro`:
- Positive with `does_not_read` (~0.6): more NARP residents, more illiteracy.
- Negative with `attends_education_5_21` and `attends_higher_education` (~ –0.4 and –0.6).
- Positive with `unemployment` (~0.6).
- Negative with `stratum` (~ –0.4): higher NARP share aligns with lower stratum.

Stratum is also strongly correlated with most outcomes, which is why it is excluded from the regression and clustering steps below.

## 4. Bivariate linear regressions

Independent variable: `ethnicity_afro` (NARP share). One regression per outcome (`unemployment`, `attends_education_5_21`, `attends_higher_education`, `does_not_read`). Interactive scatter plots colour points by stratum (Plotly figures render inline in the notebook).

In [ ]:
def fit_and_plot(df, y_col, title, y_label):
    model = LinearRegression()
    X = df[['ethnicity_afro']]
    y = df[y_col]
    model.fit(X, y)
    y_pred = model.predict(X)
    print(f'{y_col}: slope = {model.coef_[0]:.4f}, intercept = {model.intercept_:.4f}')

    stratum_order = sorted(df['stratum'].unique())
    colors = pd.factorize(df['stratum'])[0]

    fig = go.Figure(data=go.Scatter(
        x=df['ethnicity_afro'], y=df[y_col], mode='markers',
        marker=dict(
            size=8, color=colors, colorscale='aggrnyl',
            cmin=0, cmax=df['stratum'].nunique() - 1, reversescale=True,
            colorbar=dict(len=0.5, title='Stratum',
                          tickvals=np.arange(df['stratum'].nunique()),
                          ticktext=stratum_order),
        ),
        text=df['neighborhood_name'],
        hovertemplate=(
            '<b>%{text}</b><br><br>Afro share: %{x:.2f}<br>'
            + y_label + ': %{y:.2f}<br>Stratum: %{marker.color:.0f}'
        ),
    ))
    fig.add_trace(go.Scatter(
        x=df['ethnicity_afro'], y=y_pred.flatten(), mode='lines',
        line=dict(color='red', width=2),
        hovertemplate='Slope: %{y:.2f}<br>Intercept: %{x:.2f}<extra></extra>',
    ))
    fig.update_layout(
        title=title,
        xaxis_title='Afro ethnicity share (NARP)',
        yaxis_title=y_label,
        font=dict(family='Courier New, monospace', size=14, color='#7f7f7f'),
        plot_bgcolor='white',
    )
    fig.show()
    return model

### Afro share vs unemployment

In [ ]:
_ = fit_and_plot(data, 'unemployment',
                 'Afro ethnicity (NARP) vs Unemployment', 'Unemployment')

### Afro share vs education (ages 5–21)

In [ ]:
_ = fit_and_plot(data, 'attends_education_5_21',
                 'Afro ethnicity (NARP) vs Education attendance (5–21)',
                 'Attends education (5–21)')

### Afro share vs higher education

In [ ]:
_ = fit_and_plot(data, 'attends_higher_education',
                 'Afro ethnicity (NARP) vs Higher education',
                 'Attends higher education')

### Afro share vs illiteracy

In [ ]:
_ = fit_and_plot(data, 'does_not_read',
                 'Afro ethnicity (NARP) vs Illiteracy',
                 'Does not read (%)')

## 5. K-means clustering

Unsupervised grouping of neighborhoods by socioeconomic profile. Stratum is omitted (it would dominate the assignment). Features used:
- `does_not_read`
- `attends_education_5_21`
- `attends_higher_education`
- `unemployment`
- `ethnicity_afro`

k = 3 keeps the clusters large enough to be interpretable; more clusters tend to over-segment the city.

In [ ]:
features = ['does_not_read', 'attends_education_5_21',
            'attends_higher_education', 'unemployment', 'ethnicity_afro']
X = data[features].apply(pd.to_numeric, errors='coerce').dropna()
data_clustered = data.loc[X.index].copy()

k = 3
kmeans = KMeans(n_clusters=k, random_state=0, n_init=50)
kmeans.fit(X)
data_clustered['cluster'] = kmeans.labels_

print(f'KMeans converged in {kmeans.n_iter_} iterations, WSS = {kmeans.inertia_:.4f}')
print('Cluster sizes:', Counter(kmeans.labels_))

In [ ]:
cluster_means = data_clustered.groupby('cluster')[features].mean()
cluster_means

### Per-cluster densities

Each panel shows the kernel density of one feature, split by cluster. This is how the model effectively splits the city across each variable.

In [ ]:
fig = plt.figure(figsize=(20, 18))
for i, var in enumerate(features, start=1):
    ax = fig.add_subplot(math.ceil(len(features) / 2), 2, i)
    for cl, color in zip(sorted(data_clustered['cluster'].unique()), ['r', 'g', 'b']):
        sns.kdeplot(
            data_clustered.loc[data_clustered['cluster'] == cl, var],
            shade=True, color=color, ax=ax, label=f'Cluster {cl}',
        )
    ax.set_title(var)
    ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cluster_densities.png', dpi=150, bbox_inches='tight')
plt.show()

### Cluster scatter plots

Each scatter plots one socioeconomic outcome against `ethnicity_afro`, with neighborhoods coloured by their k-means cluster. Hovering shows neighborhood name and stratum.

In [ ]:
color_palette = ['red', 'green', 'blue']

for x_var, title in [
    ('does_not_read',            'Afro ethnicity vs Illiteracy'),
    ('attends_education_5_21',   'Afro ethnicity vs Education (5–21)'),
    ('unemployment',             'Afro ethnicity vs Unemployment'),
    ('attends_higher_education', 'Afro ethnicity vs Higher education'),
]:
    fig = px.scatter(
        data_clustered, x=x_var, y='ethnicity_afro', color='cluster',
        hover_data=['neighborhood_name', 'stratum'],
        color_continuous_scale=color_palette,
        width=700, height=700,
    )
    fig.update_traces(marker=dict(size=10, opacity=0.8))
    fig.update_layout(title=title)
    fig.show()

## 6. Maps

Static maps produced with QGIS from the clustered dataset. Stored as static PNGs in `figures/`.

In [ ]:
from IPython.display import Image, display

for name in ['afro_population_map.png', 'cluster_with_afro_population.png']:
    path = FIGURES_DIR / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))

## 7. Conclusion

There is a consistent relationship between NARP self-identification share and access to fundamental rights in Cali: higher Afro share aligns with higher unemployment and illiteracy, and lower school and university attendance. The k-means model groups the city into three brackets where the most disadvantaged cluster largely overlaps with neighborhoods of high Afro population (the *Aguablanca* district and surroundings), which is consistent with the *blackpolis* framework proposed by Alves (2020).

The policy brief in `README.md` uses these results to evaluate three courses of action and recommends a labor-introduction and personal-finances program targeted at the most disadvantaged cluster.

## References

- Alves, J. A. (2020). Biopólis, necrópolis, ‘blackpolis’: Notas para un nuevo léxico político en los análisis socio-espaciales del racismo. *Geopauta*, 4(1), 5. https://doi.org/10.22481/rg.v4i1.6161
- DANE. (2018). *Censo Nacional de Población y Vivienda 2018.* https://microdatos.dane.gov.co/index.php/catalog/643
- DANE. (2018). *Gran Encuesta Integrada de Hogares (GEIH).* https://microdatos.dane.gov.co/index.php/catalog/547